# TP1 — Préparation des données : Approche Data Marketing

**Philosophie** : Contrairement à une approche ML classique qui supprime les valeurs nulles ou aberrantes, nous adoptons une approche **Data Marketing**. On ne supprime aucune donnée financière (pour ne pas fausser le CA global). On se contente de les **flagger**. On ne crée une vue filtrée qu'à la fin, uniquement pour la segmentation CRM.

**Fichiers** : `transactions_bis.csv` (brut) · `customers_bis.csv` (brut)

In [ ]:
## 1. Chargement et déduplication

transactions_bis.csv : 1,837,137 lignes × 8 colonnes
customers.csv        : 50,000 lignes × 9 colonnes


In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# --- Chargement ---
transactions = pd.read_csv('data/transactions_bis.csv')
customers = pd.read_csv('data/customers_bis.csv')

print(f"transactions_bis.csv : {transactions.shape[0]:,} lignes × {transactions.shape[1]} colonnes")
print(f"customers_bis.csv    : {customers.shape[0]:,} lignes × {customers.shape[1]} colonnes")

# --- Déduplication (seule vraie suppression autorisée) ---
n_dupes = transactions.duplicated().sum()
transactions = transactions.drop_duplicates()
print(f"\nDoublons exacts supprimés : {n_dupes:,}")
print(f"Transactions après dédup : {len(transactions):,}")

In [ ]:
## 2. Création des flags sur les transactions

On ne supprime rien. On étiquette chaque ligne selon sa nature pour pouvoir filtrer à la demande.

Lignes sans customer_id : 418,258 (22.8%)

--- Profil AVEC customer_id ---
         quantity  unit_price
count  1406456.00  1418879.00
mean        12.66        3.34
std        150.97       53.58
min     -80995.00        0.00
25%          2.00        1.25
50%          4.00        1.95
75%         12.00        3.75
max      80995.00    38970.00

--- Profil SANS customer_id ---
        quantity  unit_price
count  414494.00   418258.00
mean        6.34        5.82
std        86.63      170.98
min     -9600.00   -53594.36
25%         1.00        1.63
50%         2.00        2.57
75%         5.00        4.30
max     10200.00    25111.09

Top 5 pays AVEC customer_id :
country
United Kingdom    1282213
EIRE                30461
Germany             29143
France              21812
Netherlands          7154
Name: count, dtype: int64

Top 5 pays SANS customer_id :
country
United Kingdom    399616
EIRE                5882
Germany             3307
France              2326
Belgium              720
Na

In [ ]:
# --- Flags booléens ---

# 1. Ventes anonymes (pas de customer_id)
transactions['is_anonymous'] = transactions['customer_id'].isna()

# 2. Retours / annulations (quantity <= 0)
transactions['is_return'] = transactions['quantity'] <= 0

# 3. Prix nul ou négatif
transactions['is_zero_price'] = transactions['unit_price'] <= 0

# 4. Codes non-produits (ne match PAS le format standard ^\d+[A-Za-z]?$)
transactions['is_non_product'] = ~transactions['product_code'].str.match(r'^\d+[A-Za-z]?$', na=False)

# --- Calcul du line_total ---
transactions['line_total'] = transactions['quantity'] * transactions['unit_price']

# --- Résumé des flags ---
print("Résumé des flags :")
print(f"  is_anonymous   : {transactions['is_anonymous'].sum():>8,} lignes ({100*transactions['is_anonymous'].mean():.1f}%)")
print(f"  is_return      : {transactions['is_return'].sum():>8,} lignes ({100*transactions['is_return'].mean():.1f}%)")
print(f"  is_zero_price  : {transactions['is_zero_price'].sum():>8,} lignes ({100*transactions['is_zero_price'].mean():.1f}%)")
print(f"  is_non_product : {transactions['is_non_product'].sum():>8,} lignes ({100*transactions['is_non_product'].mean():.1f}%)")
print(f"\nTotal transactions (toutes conservées) : {len(transactions):,}")
print(f"Colonnes : {list(transactions.columns)}")

IDs fictifs attribués : 110,589 factures anonymes
Lignes customer_id NaN restantes : 0
Lignes anonymes (is_anonymous=True) : 418,258
Total lignes conservées : 1,837,137


## 3. Création de la vue CRM

On filtre les transactions pour ne garder que les ventes propres (client identifié, quantité positive, prix positif, vrai produit). Ce DataFrame `transactions_crm` servira à la segmentation RFM.

In [ ]:
# Vue CRM : exclut toutes les lignes flaggées
mask_clean = (
    ~transactions['is_anonymous']
    & ~transactions['is_return']
    & ~transactions['is_zero_price']
    & ~transactions['is_non_product']
)

transactions_crm = transactions[mask_clean].copy()
transactions_crm['customer_id'] = transactions_crm['customer_id'].astype(int)

print(f"Transactions totales (brutes dédupliquées) : {len(transactions):,}")
print(f"Transactions CRM (vue filtrée)             : {len(transactions_crm):,}")
print(f"Lignes exclues de la vue CRM               : {len(transactions) - len(transactions_crm):,} ({100*(len(transactions) - len(transactions_crm))/len(transactions):.1f}%)")
print(f"Clients uniques dans la vue CRM            : {transactions_crm['customer_id'].nunique():,}")

Quantités ≤ 0 au total       : 23,314
  dont factures 'C' (annulations) : 19,494
  dont qty < 0 hors 'C'           : 3,821
  dont qty NaN                     : 16,187

Après suppression qty ≤ 0 : 1,797,636 lignes (−39,501)


## 4. Traitement du fichier clients — Insights Marketing

On ne supprime aucun outlier : les gros dépenseurs sont nos VIP. On les flag.

In [ ]:
# --- Flag VIP (outliers hauts de total_spent) ---
q1 = customers['total_spent'].quantile(0.25)
q3 = customers['total_spent'].quantile(0.75)
iqr = q3 - q1
upper_fence = q3 + 1.5 * iqr

customers['is_vip_outlier'] = customers['total_spent'] > upper_fence

print(f"IQR total_spent : Q1={q1:.2f}, Q3={q3:.2f}, IQR={iqr:.2f}")
print(f"Borne haute VIP : {upper_fence:.2f}")
print(f"Clients VIP identifiés : {customers['is_vip_outlier'].sum():,} ({100*customers['is_vip_outlier'].mean():.1f}%)")

# --- Flag One-shot (clients à 1 seule commande) ---
customers['is_one_shot'] = customers['n_orders'].round() <= 1

print(f"\nClients one-shot : {customers['is_one_shot'].sum():,} ({100*customers['is_one_shot'].mean():.1f}%)")
print(f"Total clients    : {len(customers):,}")

unit_price < 0 :
        invoice_id product_code     product_name  quantity  unit_price
81454      A563186            B  Adjust bad debt       1.0   -11062.06
553841     A516228            B  Adjust bad debt       1.0   -44031.79
1143162    A528059            B  Adjust bad debt       1.0   -38925.87
1179724    A506401            B  Adjust bad debt       1.0   -53594.36
1618681    A563187            B  Adjust bad debt       1.0   -11062.06

unit_price == 0 : 7,131 lignes
Exemples de product_code concernés :
product_code
85123A    35
85099B    21
79321     16
22501     15
21116     15
85099F    14
23084     14
22734     14
35965     13
22366     12
Name: count, dtype: int64

Après suppression unit_price ≤ 0 : 1,790,500 lignes (−7,136)


## 5. Bilan Business

In [ ]:
ca_brut = transactions['line_total'].sum()
ca_crm = transactions_crm['line_total'].sum()
n_clients_crm = transactions_crm['customer_id'].nunique()
n_vip = customers['is_vip_outlier'].sum()
n_one_shot = customers['is_one_shot'].sum()

print("=" * 60)
print("         BILAN BUSINESS — DATA MARKETING")
print("=" * 60)
print()
print(f"  💰 CA Brut Global (toutes transactions) : {ca_brut:>12,.2f} €")
print(f"  💰 CA Net CRM (vue filtrée)             : {ca_crm:>12,.2f} €")
print(f"  📉 Écart Brut vs CRM                    : {ca_brut - ca_crm:>12,.2f} € ({100*(ca_brut - ca_crm)/ca_brut:.1f}%)")
print()
print(f"  👥 Clients dans la base CRM             : {n_clients_crm:>8,}")
print(f"  ⭐ Clients VIP (outliers total_spent)    : {n_vip:>8,}")
print(f"  🎯 Clients One-shot (cible réactivation) : {n_one_shot:>8,}")
print()
print("=" * 60)
print("  Aucune donnée financière n'a été supprimée.")
print("  Les anomalies sont flaggées, pas éliminées.")
print("  La vue CRM est un filtre, pas une suppression.")
print("=" * 60)

Lignes avec codes non-produits : 6,138

Détail :
product_code
POST            2045
DOT             1634
M               1071
BANK CHARGES     229
ADJUST           223
D                193
AMAZONFEE        191
CRUK             189
B                184
S                179
Name: count, dtype: int64

Après suppression des non-produits : 1,784,362 lignes (−6,138)
